# 01. Exploratory Data Analysis (EDA)
## Human Activity Recognition using Smartphone Sensors

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config
from src.data_loader import load_raw_data, perform_data_quality_checks
from src.evaluate import generate_class_distribution_plot


### 1. Load UCI HAR Dataset and Check Quality

In [ ]:
X_train, y_train, sub_train, X_test, y_test, sub_test, feature_names = load_raw_data()
checks = perform_data_quality_checks(X_train, y_train, X_test, y_test)
print("Data Checks Summary:", checks)


### 2. Class Distribution and Subject Distribution

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(x=y_train, palette='viridis', order=config.ACTIVITY_NAMES)
plt.title("Training Set Class Distribution")
plt.xlabel("Activity")
plt.ylabel("Sample Count")
plt.show()

plt.figure(figsize=(12, 4))
sns.countplot(x=sub_train, palette='Blues_r')
plt.title("Samples per Subject (Training Set)")
plt.xlabel("Subject ID")
plt.ylabel("Sample Count")
plt.show()


### 3. Per-Activity Feature Mean Analysis

In [ ]:
top_features = ['tBodyAcc-mean()-X', 'tGravityAcc-mean()-X', 'tBodyGyro-mean()-X', 'tBodyAccJerk-mean()-X']
df_top = X_train[top_features].copy()
df_top['Activity'] = y_train.values

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, feat in enumerate(top_features):
    sns.boxplot(data=df_top, x='Activity', y=feat, ax=axes[i], palette='Set2')
    axes[i].set_title(f"Distribution of {feat}")
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


### 4. Dimensionality Reduction (PCA 2D Projection)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

pca_2d = PCA(n_components=2, random_state=config.RANDOM_SEED)
X_pca_2d = pca_2d.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
df_pca['Activity'] = y_train.values

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Activity', palette=config.ACTIVITY_COLORS, alpha=0.7)
plt.title("2D PCA Projection: Dynamic vs Static Activity Separation")
plt.grid(True, alpha=0.3)
plt.show()
